In [4]:
import pandas as pd
import numpy as np
import os
import glob

def process_batch_mit_bih(input_folder='.', output_filename='Revisi_dataset_FULL.csv'):
    print(f"--- MEMULAI PROSES BATCH DARI FOLDER: {input_folder} ---")

    # 1. Cari semua file CSV di folder (misal: 100.csv, 101.csv, dst)
    # Kita cari yang namanya angka saja (biar gak ketukar sama file lain)
    # Pola: *.csv
    all_files = glob.glob(os.path.join(input_folder, "*.csv"))

    # Filter: Ambil yang punya pasangan .txt anotasi saja
    valid_pairs = []
    for csv_file in all_files:
        base_name = os.path.splitext(csv_file)[0] # misal: ./100
        txt_file = base_name + "annotations.txt"  # misal: ./100annotations.txt

        # Cek apakah file anotasi pasangannya ada?
        if os.path.exists(txt_file):
            valid_pairs.append((csv_file, txt_file))

    print(f"Ditemukan {len(valid_pairs)} pasang file siap proses.")

    if len(valid_pairs) == 0:
        print("Tidak ada pasangan file .csv dan .txt yang cocok.")
        return

    # Penampung semua data dari semua file
    all_data_rows = []

    # Setting Window
    window_before = 90
    window_after = 98
    total_window = window_before + window_after # 188

    # 2. LOOPING SETIAP FILE
    for csv_path, txt_path in valid_pairs:
        filename = os.path.basename(csv_path)
        print(f"Sedang memproses: {filename} ...")

        try:
            # --- Load Signal ---
            sig_df = pd.read_csv(csv_path)
            sig_df.columns = [c.strip().replace("'", "") for c in sig_df.columns]

            # Cari kolom MLII (biasanya index 1, tapi kita cek namanya)
            if 'MLII' in sig_df.columns:
                signal = sig_df['MLII'].values
            elif len(sig_df.columns) > 1:
                # Fallback: ambil kolom ke-2 kalau nama bukan MLII
                signal = sig_df.iloc[:, 1].values
            else:
                print(f"  -> Skip {filename}: Kolom sinyal tidak valid.")
                continue

            # --- Load Annotation ---
            ann_df = pd.read_csv(txt_path, delim_whitespace=True)

            # Deteksi nama kolom di txt (kadang beda-beda)
            # Prioritas: Cari kolom 'Sample' dan '#'
            loc_col = 'Sample' if 'Sample' in ann_df.columns else ann_df.columns[1]
            type_col = '#' if '#' in ann_df.columns else ann_df.columns[2]

            locations = ann_df[loc_col].values
            types = ann_df[type_col].values

            # --- Potong & Labeling ---
            count_local = 0
            for r_peak, beat_type in zip(locations, types):
                start = r_peak - window_before
                end = r_peak + window_after

                if start < 0 or end > len(signal):
                    continue

                segment = signal[start:end]

                # Labeling
                label = -1
                if beat_type in ['N', 'L', 'R', 'e', 'j']:
                    label = 0 # Normal
                elif beat_type in ['A', 'a', 'J', 'S', 'V', 'E', 'F']:
                    label = 1 # Abnormal

                if label != -1:
                    row = list(segment)
                    row.append(label)
                    all_data_rows.append(row)
                    count_local += 1

            print(f"  -> {filename}: Berhasil ambil {count_local} sampel.")

        except Exception as e:
            print(f"  -> Error pada {filename}: {e}")

    # 3. SIMPAN HASIL GABUNGAN
    print(f"--- MENGGABUNGKAN DATA ---")
    if len(all_data_rows) > 0:
        cols = [f'f{i}' for i in range(total_window)] + ['label']
        df_final = pd.DataFrame(all_data_rows, columns=cols)

        # Simpan ke CSV (Pakai titik koma ; sesuai request sebelumnya)
        df_final.to_csv(output_filename, index=False, sep=';')

        print(f"SUKSES TOTAL!")
        print(f"Dataset tersimpan di: {output_filename}")
        print(f"Total Sampel Gabungan: {len(df_final)} baris")
    else:
        print("Gagal: Tidak ada data valid yang terkumpul.")


process_batch_mit_bih(input_folder='.', output_filename='REvisi_dataset_FULL.csv')

--- MEMULAI PROSES BATCH DARI FOLDER: . ---
Ditemukan 1 pasang file siap proses.
Sedang memproses: 100.csv ...


/tmp/ipython-input-1828964916.py:59: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  ann_df = pd.read_csv(txt_path, delim_whitespace=True)


  -> 100.csv: Berhasil ambil 2271 sampel.
--- MENGGABUNGKAN DATA ---
SUKSES TOTAL!
Dataset tersimpan di: REvisi_dataset_FULL.csv
Total Sampel Gabungan: 2271 baris
